# Lab 03.1: Quantization Deep Dive**INT8/INT4/FP16 comparison | GPTQ vs AWQ vs bitsandbytes | Perplexity-Speed Tradeoffs | Memory Savings**This notebook implements quantization from scratch, benchmarks precision methods,and measures the perplexity-speed-memory tradeoff space.

In [ ]:
import syssys.path.insert(0, '../../..')import torchimport numpy as npimport matplotlib.pyplot as pltfrom dataclasses import dataclassfrom typing import Dict, Tuple, Listimport timefrom utils import benchmark, gpu_infofrom utils.benchmark import time_cuda, BenchmarkResultfrom utils.gpu_info import detect_gpu, print_gpu_info, GPU_CATALOGplt.style.use('seaborn-v0_8-darkgrid')%matplotlib inlineDEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'print(f'Device: {DEVICE}')if DEVICE == 'cuda':    gpu = detect_gpu()    print_gpu_info(gpu)

## 1. Core Quantization ImplementationsSymmetric per-tensor quantization at different bit widths. Error ∝ `max_val / (2^bits - 1)` — outliers destroy precision.

In [ ]:
def quantize_symmetric(tensor: torch.Tensor, bits: int) -> torch.Tensor:    qmax = 2 ** (bits - 1) - 1    scale = tensor.abs().max() / qmax    quantized = torch.clamp(torch.round(tensor / scale), -qmax - 1, qmax)    return quantized * scaledef quantize_fp16(t): return t.to(torch.float16).to(torch.float32)def quantize_int8(t): return quantize_symmetric(t, 8)def quantize_int4(t): return quantize_symmetric(t, 4)def quantize_nf4(tensor: torch.Tensor) -> torch.Tensor:    nf4_levels = torch.tensor([        -1.0, -0.6962, -0.5251, -0.3949, -0.2844, -0.1848, -0.0911, 0.0,        0.0796, 0.1609, 0.2461, 0.3379, 0.4407, 0.5626, 0.7230, 1.0])    abs_max = tensor.abs().max()    if abs_max == 0: return tensor.clone()    normalized = tensor / abs_max    indices = (normalized.unsqueeze(-1) - nf4_levels).abs().argmin(dim=-1)    return nf4_levels[indices] * abs_maxdef measure_error(original, quantized):    diff = original - quantized    mse = (diff ** 2).mean().item()    signal = (original ** 2).mean().item()    return {'mse': mse, 'mae': diff.abs().mean().item(),            'max_error': diff.abs().max().item(),            'sqnr_db': 10 * np.log10(signal / max(mse, 1e-10))}

## 2. INT8 vs INT4 vs FP16 Precision Comparison

In [ ]:
torch.manual_seed(42)weights = torch.randn(4096, 4096) * 0.02  # LLM weight distributionmethods = {'FP16': quantize_fp16, 'INT8': quantize_int8,           'INT4': quantize_int4, 'NF4': quantize_nf4}print(f"{'Method':<8} {'MSE':>12} {'MAE':>12} {'SQNR (dB)':>12}")print('-' * 48)results = {}for name, fn in methods.items():    m = measure_error(weights, fn(weights))    results[name] = m    print(f"{name:<8} {m['mse']:>12.2e} {m['mae']:>12.2e} {m['sqnr_db']:>12.1f}")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))for ax, (name, fn) in zip(axes, methods.items()):    err = (weights - fn(weights)).flatten().numpy()    ax.hist(err, bins=100, alpha=0.7, color='steelblue')    ax.set_title(f'{name} Error'); ax.set_xlabel('Error')    ax.axvline(0, color='red', ls='--', alpha=0.5)plt.tight_layout(); plt.show()

## 3. GPTQ vs AWQ vs bitsandbytes SimulationGPTQ: calibration-based weight rounding (Hessian-guided).AWQ: activation-aware — protects salient channels.bitsandbytes: blockwise NF4 with double quantization.

In [ ]:
def gptq_simulate(tensor: torch.Tensor, group_size: int = 128) -> torch.Tensor:    """GPTQ-style: per-group INT4 with Hessian-guided rounding order."""    rows, cols = tensor.shape    result = tensor.clone()    for i in range(0, cols, group_size):        group = tensor[:, i:i+group_size]        scale = group.abs().max() / 7.0        q = torch.clamp(torch.round(group / scale), -8, 7)        # Simulate error compensation (simplified OBQ)        err = group - q * scale        if i + group_size < cols:            result[:, i+group_size:i+2*group_size] += err.mean(dim=1, keepdim=True) * 0.1        result[:, i:i+group_size] = q * scale    return resultdef awq_simulate(tensor: torch.Tensor, salient_frac: float = 0.01) -> torch.Tensor:    """AWQ-style: protect top salient channels with higher precision."""    col_importance = tensor.abs().mean(dim=0)    k = int(tensor.shape[1] * salient_frac)    _, top_idx = col_importance.topk(k)    result = quantize_int4(tensor)    result[:, top_idx] = quantize_fp16(tensor[:, top_idx])  # keep salient in FP16    return resultdef bnb_nf4_simulate(tensor: torch.Tensor, block_size: int = 64) -> torch.Tensor:    """bitsandbytes-style blockwise NF4."""    flat = tensor.flatten()    out = torch.zeros_like(flat)    for i in range(0, len(flat), block_size):        block = flat[i:i+block_size]        out[i:i+block_size] = quantize_nf4(block)    return out.reshape(tensor.shape)# Benchmark all methodstorch.manual_seed(42)W = torch.randn(4096, 4096) * 0.02quant_methods = {    'FP16': quantize_fp16, 'INT8': quantize_int8,    'INT4 (naive)': quantize_int4, 'NF4 (bitsandbytes)': bnb_nf4_simulate,    'GPTQ (group=128)': gptq_simulate, 'AWQ (1% salient)': awq_simulate,}print(f"{'Method':<22} {'MSE':>10} {'SQNR(dB)':>10} {'Bits':>6}")print('-' * 52)method_results = {}for name, fn in quant_methods.items():    m = measure_error(W, fn(W))    method_results[name] = m    bits = 16 if 'FP16' in name else (8 if 'INT8' in name else 4)    print(f"{name:<22} {m['mse']:>10.2e} {m['sqnr_db']:>10.1f} {bits:>6}")

In [ ]:
names = list(method_results.keys())sqnrs = [method_results[n]['sqnr_db'] for n in names]fig, ax = plt.subplots(figsize=(10, 4))colors = ['#2563eb', '#059669', '#dc2626', '#7c3aed', '#d97706', '#0891b2']bars = ax.barh(names, sqnrs, color=colors)ax.set_xlabel('SQNR (dB) — higher is better')ax.set_title('Quantization Quality: GPTQ vs AWQ vs bitsandbytes vs Naive')ax.axvline(x=40, color='gray', ls='--', alpha=0.5, label='Good threshold')ax.legend(); plt.tight_layout(); plt.show()

## 4. Perplexity vs Speed TradeoffSimulate language model projection to measure perplexity degradation under quantization.

In [ ]:
def compute_perplexity(logits, ref_probs):    log_probs = torch.log_softmax(logits, dim=-1)    ce = -(ref_probs * log_probs).sum(dim=-1).mean()    return torch.exp(ce).item()torch.manual_seed(123)hidden = torch.randn(256, 2048) * 0.5proj_w = torch.randn(32000, 2048) * (2.0 / 2048) ** 0.5ref_logits = hidden @ proj_w.Tref_probs = torch.softmax(ref_logits, dim=-1)ref_ppl = compute_perplexity(ref_logits, ref_probs)ppl_methods = {'FP32': lambda w: w, 'FP16': quantize_fp16, 'INT8': quantize_int8,               'INT4': quantize_int4, 'NF4': quantize_nf4,               'GPTQ': gptq_simulate, 'AWQ': awq_simulate}print(f"{'Method':<12} {'PPL':>10} {'ΔPPL':>10} {'Degradation':>12}")print('-' * 48)ppl_results = {}for name, fn in ppl_methods.items():    qw = fn(proj_w)    ppl = compute_perplexity(hidden @ qw.T, ref_probs)    delta = ppl - ref_ppl    ppl_results[name] = {'ppl': ppl, 'delta': delta, 'pct': delta/ref_ppl*100}    print(f"{name:<12} {ppl:>10.2f} {delta:>+10.2f} {delta/ref_ppl*100:>+11.2f}%")

In [ ]:
# Measure quantization throughput (time to quantize + matmul)torch.manual_seed(42)X = torch.randn(256, 2048)W_bench = torch.randn(2048, 2048) * 0.02speed_results = {}for name, fn in [('FP16', quantize_fp16), ('INT8', quantize_int8),                 ('INT4', quantize_int4), ('GPTQ', gptq_simulate), ('AWQ', awq_simulate)]:    t0 = time.perf_counter()    for _ in range(10):        qw = fn(W_bench)        _ = X @ qw.T    elapsed = (time.perf_counter() - t0) / 10 * 1000    speed_results[name] = elapsedprint(f"{'Method':<12} {'Latency (ms)':>14}")print('-' * 28)for n, t in speed_results.items():    print(f"{n:<12} {t:>14.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))for name in ['FP16', 'INT8', 'INT4', 'GPTQ', 'AWQ']:    if name in ppl_results and name in speed_results:        ax.scatter(speed_results[name], ppl_results[name]['pct'],                   s=120, zorder=5, label=name)        ax.annotate(name, (speed_results[name], ppl_results[name]['pct']),                    textcoords='offset points', xytext=(8, 4))ax.set_xlabel('Latency (ms) — lower is better')ax.set_ylabel('Perplexity Degradation (%) — lower is better')ax.set_title('Pareto Frontier: Speed vs Quality')ax.axhline(y=1.0, color='green', ls='--', alpha=0.5, label='1% threshold')ax.axhline(y=5.0, color='red', ls='--', alpha=0.5, label='5% threshold')ax.legend(); plt.tight_layout(); plt.show()

## 5. Memory Savings MeasurementCalculate exact memory footprint for popular model sizes across quantization levels.

In [ ]:
@dataclassclass ModelSpec:    name: str    params_b: float    hidden: int    layers: int    heads: intmodels = [ModelSpec('LLaMA-7B', 7, 4096, 32, 32),          ModelSpec('LLaMA-13B', 13, 5120, 40, 40),          ModelSpec('LLaMA-70B', 70, 8192, 80, 64),          ModelSpec('Mixtral-8x7B', 46.7, 4096, 32, 32)]def mem_gb(params_b, bits): return params_b * 1e9 * bits / 8 / 1024**3print(f"{'Model':<14} {'FP32':>7} {'FP16':>7} {'INT8':>7} {'INT4':>7} {'Savings':>9}")print(f"{'':14} {'(GB)':>7} {'(GB)':>7} {'(GB)':>7} {'(GB)':>7} {'(vs FP16)':>9}")print('-' * 56)mem_data = []for m in models:    fp32, fp16, i8, i4 = [mem_gb(m.params_b, b) for b in [32, 16, 8, 4]]    savings = (1 - i4/fp16) * 100    mem_data.append({'name': m.name, 'fp16': fp16, 'int8': i8, 'int4': i4})    print(f"{m.name:<14} {fp32:>7.1f} {fp16:>7.1f} {i8:>7.1f} {i4:>7.1f} {savings:>8.0f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))x = np.arange(len(mem_data))w = 0.25for i, (bits, key, color) in enumerate([('FP16','fp16','#3b82f6'),                                          ('INT8','int8','#10b981'),                                          ('INT4','int4','#f59e0b')]):    vals = [d[key] for d in mem_data]    ax.bar(x + i*w, vals, w, label=bits, color=color)ax.set_xticks(x + w); ax.set_xticklabels([d['name'] for d in mem_data])ax.set_ylabel('Memory (GB)'); ax.set_title('Model Memory by Quantization Level')ax.legend(); plt.tight_layout(); plt.show()

## 6. KV Cache Quantization & TurboQuantRotation-based 3-bit KV cache compression — spreads outliers uniformly before quantization.

In [ ]:
def random_rotation(x: torch.Tensor) -> torch.Tensor:    dim = x.shape[-1]    torch.manual_seed(7)    Q, _ = torch.linalg.qr(torch.randn(dim, dim))    return x @ Qdef inverse_rotation(x: torch.Tensor) -> torch.Tensor:    dim = x.shape[-1]    torch.manual_seed(7)    Q, _ = torch.linalg.qr(torch.randn(dim, dim))    return x @ Q.Tdef scalar_3bit(tensor):    scale = tensor.abs().max() / 3.5    return torch.clamp(torch.round(tensor / scale), -4, 3) * scaledef turbo_quant(kv):    return inverse_rotation(scalar_3bit(random_rotation(kv)))# Simulate KV cache with outlierstorch.manual_seed(99)kv = torch.randn(32, 512, 128) * 0.3outlier_mask = torch.rand_like(kv) < 0.02kv[outlier_mask] *= 8.0flat_kv = kv.reshape(-1, 128)tq_methods = {'Direct INT4': quantize_int4, 'Direct 3-bit': scalar_3bit,              'TurboQuant 3-bit': turbo_quant}print(f"{'Method':<20} {'MSE':>10} {'SQNR(dB)':>10} {'CosSim':>10}")print('-' * 54)for name, fn in tq_methods.items():    m = measure_error(flat_kv, fn(flat_kv))    cos = torch.nn.functional.cosine_similarity(flat_kv, fn(flat_kv), dim=-1).mean().item()    print(f"{name:<20} {m['mse']:>10.2e} {m['sqnr_db']:>10.1f} {cos:>10.6f}")# Outlier analysisorig_outliers = (flat_kv.abs() > flat_kv.std() * 3).float().mean().item() * 100rotated = random_rotation(flat_kv)rot_outliers = (rotated.abs() > rotated.std() * 3).float().mean().item() * 100print(f"\nOutliers before rotation: {orig_outliers:.2f}%")print(f"Outliers after rotation:  {rot_outliers:.2f}%")print(f"\nCompression: FP16 -> 3-bit = {16/3:.1f}x memory reduction")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))sample = flat_kv[0].numpy()rotated_sample = random_rotation(flat_kv)[0].numpy()axes[0].bar(range(128), np.abs(sample), color='#ef4444', alpha=0.7)axes[0].set_title('Before Rotation (outliers concentrated)')axes[0].set_xlabel('Dimension'); axes[0].set_ylabel('|Value|')axes[1].bar(range(128), np.abs(rotated_sample), color='#10b981', alpha=0.7)axes[1].set_title('After Rotation (outliers spread)')axes[1].set_xlabel('Dimension')plt.tight_layout(); plt.show()

## 7. GPU Fit Analysis: Which Models Fit Where?

In [ ]:
gpus = {'A10G (24GB)': 24, 'RTX 4090 (24GB)': 24, 'A100 (80GB)': 80, 'H100 (80GB)': 80}print(f"{'Model':<14} {'Quant':<6} {'Size(GB)':>9} | ", end='')print(' | '.join(f"{g:<8}" for g in gpus.keys()))print('-' * 90)for m in models:    for bits, qname in [(16,'FP16'), (8,'INT8'), (4,'INT4')]:        size = mem_gb(m.params_b, bits)        fits = ['  ✅  ' if size < v * 0.85 else '  ❌  ' for v in gpus.values()]        print(f"{m.name:<14} {qname:<6} {size:>8.1f} | {'|'.join(fits)}")    print()

## 8. Quantization Overhead: Latency at Scale

In [ ]:
sizes = [512, 1024, 2048, 4096, 8192]latencies = {n: [] for n in ['FP16', 'INT8', 'INT4', 'GPTQ']}for dim in sizes:    W_test = torch.randn(dim, dim) * 0.02    for name, fn in [('FP16', quantize_fp16), ('INT8', quantize_int8),                     ('INT4', quantize_int4), ('GPTQ', gptq_simulate)]:        t0 = time.perf_counter()        for _ in range(5):            fn(W_test)        latencies[name].append((time.perf_counter() - t0) / 5 * 1000)fig, ax = plt.subplots(figsize=(8, 5))for name, vals in latencies.items():    ax.plot(sizes, vals, 'o-', label=name, linewidth=2)ax.set_xlabel('Matrix Dimension'); ax.set_ylabel('Quantization Time (ms)')ax.set_title('Quantization Overhead vs Matrix Size')ax.legend(); ax.set_yscale('log'); plt.tight_layout(); plt.show()

## 9. Quantization Decision FrameworkGiven model size, GPU memory, and quality tolerance — which method to use?

In [ ]:
def recommend(model_gb_fp16, gpu_gb, max_ppl_pct=5.0):    if gpu_gb >= model_gb_fp16 * 1.3:        return 'FP16', 'Sufficient memory, best quality'    elif gpu_gb >= model_gb_fp16 * 0.65:        return 'INT8 (W8A16)', '<1% PPL loss, 2x compression'    elif gpu_gb >= model_gb_fp16 * 0.35:        if max_ppl_pct <= 3:            return 'GPTQ-INT4 (g128)', 'Calibrated, 1-3% PPL'        return 'AWQ-INT4', 'Activation-aware, 2-5% PPL'    else:        return 'GGUF Q3_K_M', 'Extreme compression, 5-15% PPL'scenarios = [    ('LLaMA-7B', 13.0, 'A100-80GB', 80), ('LLaMA-7B', 13.0, 'RTX 4090', 24),    ('LLaMA-70B', 130.0, 'A100-80GB', 80), ('LLaMA-70B', 130.0, 'RTX 4090', 24),    ('LLaMA-70B', 130.0, '2xA100', 160),]print(f"{'Scenario':<30} {'Recommendation':<20} {'Rationale'}")print('-' * 80)for model, fp16_gb, gpu_name, gpu_gb in scenarios:    method, reason = recommend(fp16_gb, gpu_gb)    print(f"{model} on {gpu_name:<12} {method:<20} {reason}")

## 10. Summary: Quality-Memory-Speed Heatmap

In [ ]:
summary = {    'FP16':  {'quality': 10, 'memory': 2, 'speed': 8},    'INT8':  {'quality': 9,  'memory': 5, 'speed': 9},    'GPTQ':  {'quality': 7,  'memory': 8, 'speed': 7},    'AWQ':   {'quality': 8,  'memory': 8, 'speed': 7},    'NF4':   {'quality': 7,  'memory': 8, 'speed': 6},    '3-bit': {'quality': 5,  'memory': 9, 'speed': 5},}data = np.array([[v['quality'], v['memory'], v['speed']] for v in summary.values()])fig, ax = plt.subplots(figsize=(8, 5))im = ax.imshow(data, cmap='RdYlGn', aspect='auto', vmin=1, vmax=10)ax.set_xticks([0, 1, 2]); ax.set_xticklabels(['Quality', 'Memory Efficiency', 'Speed'])ax.set_yticks(range(len(summary))); ax.set_yticklabels(list(summary.keys()))for i in range(data.shape[0]):    for j in range(data.shape[1]):        ax.text(j, i, f'{data[i,j]}', ha='center', va='center', fontsize=12, fontweight='bold')plt.colorbar(im, label='Score (1-10)'); ax.set_title('Quantization Method Comparison')plt.tight_layout(); plt.show()print('\n=== Key Takeaways ===')print('• INT8: Best quality/compression ratio — use when model barely fits')print('• GPTQ: Best INT4 for static serving — calibrate once, deploy forever')print('• AWQ: Best INT4 for generation — protects attention-critical weights')print('• NF4/bitsandbytes: Best for QLoRA fine-tuning — information-optimal for normals')print('• TurboQuant 3-bit: Best KV cache compression — rotation eliminates outlier damage')